# mp-spawn-workers — ex1: launch a 2-rank distributed job with mp.spawn

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mp-spawn-workers`. Running the final beacon cell reports progress against the `Distributed: mp.spawn workers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: mp.spawn workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mp-spawn-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mp-spawn-workers"
DD_SUBTOPIC = "Distributed: mp.spawn workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: `mp.spawn`
`torch.multiprocessing.spawn(fn, args=(...), nprocs=N, join=True)` is the canonical ARENA pattern. **Gotcha:** `spawn` pickles `fn` and sends it to the child — but functions defined in `__main__` (or a notebook cell) cannot be pickled. The drill teaches the two-line workaround: write the worker to a real `.py` file, `importlib.import_module` it, then spawn that imported attribute.

### Exercise 1 — launch a 2-rank distributed job with mp.spawn

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `mp.spawn` with `nprocs=2` to launch two worker procs that each call `init_process_group(gloo)`, by writing the worker to a temp `.py` file and importing it (Colab/notebook-correct pattern).
> Keywords: mp.spawn, nprocs, join, tempfile-worker, pickling
> ```

**KCs targeted:** `mp-spawn-with-nprocs`, `spawn-worker-must-be-importable`

Implement `ex1_launch_with_spawn(port)`. The full Colab-safe `mp.spawn` recipe:

1. Use the pre-defined `WORKER_SRC` string (below in the stub) as the source of the worker function `worker(rank, world_size, port)`. It already does init/destroy.
2. Write `WORKER_SRC` to `/tmp/dd_spawn_worker.py` (text mode).
3. Add `/tmp` to `sys.path` if it's not there, then `importlib.import_module('dd_spawn_worker')` (use `importlib.reload` if it's already cached from a prior cell run).
4. Call `mp.spawn(mod.worker, args=(2, port), nprocs=2, join=True)`.
5. Return `True` on success.

**Why the tempfile dance?** `mp.spawn` pickles the function ref and ships it to the child interpreter, which then needs to *import* the module to unpickle. Cell-defined functions live in `__main__` and aren't reachable from a fresh child. Writing to a real module file is the workaround.

**ARENA cheat.** In ARENA's `.py` runner files this isn't needed — the worker is already at module scope. The dance is only required when launching from a Jupyter/Colab cell.

In [ ]:
def ex1_launch_with_spawn(port: int) -> bool:
    path = '/tmp/dd_spawn_worker.py'
    with open(path, 'w') as f:
        f.write(WORKER_SRC)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    if 'dd_spawn_worker' in sys.modules:
        mod = importlib.reload(sys.modules['dd_spawn_worker'])
    else:
        mod = importlib.import_module('dd_spawn_worker')
    mp.spawn(mod.worker, args=(2, port), nprocs=2, join=True)
    return True


<details><summary>Solution</summary>

```python
def ex1_launch_with_spawn(port: int) -> bool:
    path = '/tmp/dd_spawn_worker.py'
    with open(path, 'w') as f:
        f.write(WORKER_SRC)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    if 'dd_spawn_worker' in sys.modules:
        mod = importlib.reload(sys.modules['dd_spawn_worker'])
    else:
        mod = importlib.import_module('dd_spawn_worker')
    mp.spawn(mod.worker, args=(2, port), nprocs=2, join=True)
    return True
```

**`nprocs` vs `world_size`.** `nprocs` says how many child procs `spawn` should create. The worker fn receives `rank` as its FIRST positional arg automatically (spawn injects it); `world_size` you pass yourself via `args=(...)`. ARENA's convention: `args=(world_size,)` and `nprocs=world_size`.

**`join=True` blocks the launcher.** Without it, `mp.spawn` returns immediately and you race against the children. Always `join=True` unless you have a specific reason to detach.

**Why we accept the tempfile.** The whole point of teaching the drill on Colab is the gotcha — once you've felt the pickling error once, you remember to put worker code in `.py` files. In a real training repo, your trainer module is already importable, so the dance disappears.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()